# common

In [ ]:
///- --test static

open testing

## common

### retry_fn'

In [ ]:
inl retry_fn' retries fn =
    let rec 루프 retry =
        inl is_error, result =
            match fn () with
            | Ok x => false, x
            | Error x => true, x
        if not is_error || retry >= retries
        then result
        else
            trace Debug
                fun () => "common.retry_fn\' / 루프"
                fun () =>
                    open sm'_operators
                    { is_error retry = sm'.obj_to_string retry ++# "/" ++# sm'.obj_to_string retries; result }
            루프 (retry + 1)
    루프 1

## fsharp

### upcast

In [ ]:
inl upcast forall t u. (x : t) : u =
    $'!x :> `u '

### downcast

In [ ]:
inl downcast forall t u. (x : t) : u =
    $'!x :?> `u '

### random

In [ ]:
nominal random = $'System.Random'

inl random () : random =
    $'`random ' ()

### random_next

In [ ]:
inl random_next (min : i32) (max : i32) (random : random) : i32 =
    $'!random.Next (!min, !max)'

### disposable

In [ ]:
type disposable_switch t =
    {
        Gleam : $'Nil'
        Fsharp : $'System.IDisposable'
        Python : $'object'
        Rust : $'Rc<dyn Fn()>'
        TypeScript : $'() => void'
    }
nominal disposable t = $'backend_switch `(disposable_switch t)'

### dispose

In [ ]:
inl dispose (disposable : disposable _) : () =
    backend_switch {
        Gleam = fun () => $'// common.dispose' : ()
        Lua = fun () => $'-- common.dispose' : ()
        Fsharp = fun () => disposable |> $'_.Dispose()' : ()
        Python = fun () => $'!disposable.__exit__(None, None, None)' : ()
        Rust = fun () => $'(!disposable)()' : ()
        TypeScript = fun () => $'(!disposable)()' : ()
    }

### yield

In [ ]:
inl yield forall t. (x : t) : () =
    $'yield !x '

### return

In [ ]:
inl return forall t. (x : t) : () =
    $'return !x '

### return'

In [ ]:
inl return' forall t. (x : t) : t =
    $'return !x '

### return_in

In [ ]:
inl return_in forall t. (x : t) : t =
    $'!x in return !x '

### retry_fn

In [ ]:
inl retry_fn forall t. retries (fn : () -> t) : option t =
    let rec 루프 retry =
        try
            fun () =>
                if retry < retries
                then fn () |> Some
                else None
            fun ex =>
                trace Warning
                    fun () => "common.retry_fn"
                    fun () => { retry ex }
                None
        |> function
            | Some x => x
            | None => 루프 (retry + 1)
    루프 0

In [1]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

inl retry_fn_test = mut 0i32
fun () =>
    retry_fn_test <- *retry_fn_test + 1
    *retry_fn_test
|> retry_fn 3i32
|> _assert_eq' (Some 1i32)

.fsx (fsharp)
__assert_eq' { expected = Some(1) }
.py (cuda)
__assert_eq' { expected = Some(1) }
.rs (rust)
__assert_eq' { expected = Some(1) }
.ts (typescript)
__assert_eq' { expected = Some(1) }

0

In [2]:
///- --test
///> fsharp
///> rust
///> typescript
///> cuda

inl retry_fn_test = mut 0i32
fun () =>
    if *retry_fn_test >= 2
    then *retry_fn_test
    else
        retry_fn_test <- *retry_fn_test + 1
        failwith "test"
|> retry_fn 3i32
|> _assert_eq' (Some 2i32)

.fsx (fsharp)
15:09:40 1w common.retry_fn / { retry = 0; ex = System.Exception: test
   at FSI_0001.closure1(Mut0 v0, Int32 v1, Unit unitVar2) in /home/runner/work/spiral/spiral/lib/spiral/main.fsx:line 56
   at FSI_0001.method0(Mut0 v0, Int32 v1) in /home/runner/work/spiral/spiral/lib/spiral/main.fsx:line 436 }
15:09:41 2w common.retry_fn / { retry = 1; ex = System.Exception: test
   at FSI_0001.closure1(Mut0 v0, Int32 v1, Unit unitVar2) in /home/runner/work/spiral/spiral/lib/spiral/main.fsx:line 56
   at FSI_0001.method0(Mut0 v0, Int32 v1) in /home/runner/work/spiral/spiral/lib/spiral/main.fsx:line 436 }
__assert_eq' { expected = Some(2) }
.rs (rust)
15:09:46 1w common.retry_fn / { retry = 0; ex = "test" }
15:09:46 2w common.retry_fn / { retry = 1; ex = "test" }
__assert_eq' { expected = Some(2) }
.ts (typescript)
15:09:48 1w common.retry_fn / { retry = 0; ex = Error: test }
15:09:48 2w common.retry_fn / { retry = 1; ex = Error: test }
__assert_eq' { expected = Some(2) }
.py (cuda)
1

0

## common

### random'

In [ ]:
inl random' forall t. (min : t) (max : t) : t =
    run_target function
        | Zig => fun () =>
            inl x = guid.zig_random_u64 ()
            $'(blk: { const lo: i128 = !min; const span: i128 = \@as(i128, !max) - lo; const n: i128 = if (span <= 0) 0 else \@intCast(\@as(u128, !x) % \@as(u128, \@intCast(span))); break :blk \@as(`t, \@intCast(lo + n)); })' : t
        | Rust => fun () =>
            open rust.rust_operators
            !\\((min, max), $'"{ use std::sync::atomic::{AtomicU64, Ordering}; use std::hash::{BuildHasher, Hasher}; static SPIRAL_NATIVE_RNG: AtomicU64 = AtomicU64::new(0); let mut x = SPIRAL_NATIVE_RNG.load(Ordering::Relaxed); if x == 0 { let mut h = std::collections::hash_map::RandomState::new().build_hasher(); h.write_u128(std::time::SystemTime::now().duration_since(std::time::UNIX_EPOCH).map(|d| d.as_nanos()).unwrap_or(0)); x = h.finish() | 1; } x ^= x << 13; x ^= x >> 7; x ^= x << 17; SPIRAL_NATIVE_RNG.store(x, Ordering::Relaxed); let span = ($1 as i128) - ($0 as i128); let n = if span <= 0 { 0 } else { (x as i128) % span }; (($0 as i128) + n) as _ }"')
        | Cuda _ => fun () =>
            global "import random"
            $'random.randrange(!min, !max)' : t
        | TypeScript => fun () =>
            $'(() => { const a = Number(!min), b = Number(!max); const r = a + Math.floor(Math.random() * Math.max(0, b - a)); return (typeof !min === "bigint" ? BigInt(r) : r); })()' : t
        | _ => fun () =>
            random () |> random_next (convert.i32 min) (convert.i32 max) |> convert.convert

### new_disposable

In [ ]:
inl new_disposable (fn : () -> ()) : disposable _ =
    run_target function
        | Rust => fun () =>
            open rust.rust_operators
            !\\(fn, $'"$0.clone()"') : disposable _
        | TypeScript => fun () =>
            inl fn = join fn
            $'!fn ' : disposable _
        | Fsharp _ => fun () =>
            inl fn = join fn
            $'{ new System.IDisposable with member _.Dispose () = !fn () }'
        | Cuda _ => fun () =>
            $'class Disposable:'
            $'    def __init__(self, fn):'
            $'        self.fn = fn'
            $'    def __exit__(self, exc_type, exc_value, traceback):'
            $'        self.fn()'
            $'        return False'
            $'Disposable(!fn)'
        | _ => fun () => null ()

In [3]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

inl new_disposable_test = mut 0i32
new_disposable fun () => new_disposable_test <- *new_disposable_test + 1
|> fun x => x : disposable ()
|> dispose
*new_disposable_test |> _assert_eq 1

.fsx (fsharp)
__assert_eq { expected = 1 }
.py (cuda)
__assert_eq { expected = 1 }
.rs (rust)
__assert_eq { expected = 1 }
.ts (typescript)
__assert_eq { expected = 1 }

0

In [4]:
///- --test
///> fsharp

inl new_disposable_test = mut 0i32
fun () =>
    new_disposable fun () => new_disposable_test <- *new_disposable_test + 1
    |> fun x => x : disposable ()
    |> use
    |> ignore
    |> return
|> async.new_task
|> async.await_task
|> async.run_synchronously
*new_disposable_test |> _assert_eq 1

__assert_eq { expected = 1 }

In [5]:
///- --test
///> fsharp

inl new_disposable_test = mut 0i32
fun () =>
    new_disposable fun () => new_disposable_test <- *new_disposable_test + 1
    |> fun x => x : disposable ()
    |> ignore
    |> return
|> async.new_task
|> async.await_task
|> async.run_synchronously
*new_disposable_test |> _assert_eq 0

__assert_eq { expected = 0 }

## main

In [6]:
///> fsharp

inl main () =
    init_trace_state None
    inl new_disposable x : _ () = new_disposable x
    $'let new_disposable x = !new_disposable x' : ()
    inl retry_fn (r : i32) (x : () -> _) : optionm'.option' () = retry_fn r x |> optionm'.box
    $'let retry_fn x = !retry_fn x' : ()
    inl memoize (fn : () -> ()) : () -> () = memoize fn
    $'let memoize x = !memoize x' : ()